# Harness Bench

**A very small benchmark to find the harness you need.**

Choose a few tasks that matter to your work, preview a local comparison and inspect saved results. Run All previews your settings and reads saved CSV files. Start inference yourself from a terminal when you are ready.

In [ ]:
from pathlib import Path
import shlex
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/local_comparison.py").is_file())
sys.path.insert(0, str(root / "scripts"))
from local_comparison import comparison_command, saved_campaigns, read_results

## Choose the experiment

The demo checks one retrieval analysis on held-out data. The quick suite covers five tasks. Each additional attempt repeats each selected task for both harnesses.

Available harness names are pi, opencode, droid and dsh. Droid and DSH local routes await model-driven validation.


In [ ]:
model = "lfm3:1.7b"
harnesses = ["pi", "opencode"]
suite = "demo"
attempts = 1

command = comparison_command(model, harnesses, suite, attempts)
print("Terminal command")
print(shlex.join(command))

## Read a campaign

After running the command yourself, rerun the next cells. Set campaign_id to a timestamp to inspect a specific experiment. An interrupted campaign can contain incomplete trials.

In [ ]:
campaign_id = None
available = saved_campaigns()
campaign = (root / "runs/campaigns" / campaign_id if campaign_id
            else next(iter(available), None))
rows = read_results(campaign) if campaign else []
print(campaign.name if campaign else "No local results yet.")
for row in rows:
    print(row["harness"], row["task"], row["status"],
          "reward", row["reward"] or "unknown",
          "seconds", row["agent_seconds"] or "unknown")

## Compare paired outcomes

Match trials by task and attempt. Review both output files when grades differ. Runtime includes hardware and model-loading effects. Run further attempts before interpreting a single gap as a repeatable difference.

In [ ]:
pairs = {}
for row in rows:
    key = (row["task"], row["attempt"])
    outcome = row["reward"] if row["status"] == "graded" else row["status"]
    pairs.setdefault(key, {})[row["harness"]] = outcome
print("task | attempt | " + " | ".join(harnesses))
for (task, attempt), outcomes in sorted(pairs.items()):
    print(" | ".join([task, attempt] + [outcomes.get(h, "missing") for h in harnesses]))